# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/baseline_public_v1.jsonl"
MAX_TOKENS  = 16384

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_HOST_IP"] = "127.0.0.1"   # avoid hostname → public-IP socket binding

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
# Phase 1 prompts — see src/cse151b_comp/prompts.py for the source of truth
# (this cell mirrors that file so the notebook stays self-contained).

_TOKEN_BUDGET_RULE = (
    "If you are running out of reasoning space, IMMEDIATELY output your "
    "best-guess answer inside \\boxed{...} before stopping. Never end "
    "without a final boxed answer."
)

_NUMERIC_PRECISION_RULE = (
    "Do not round numerical answers. Report at least 6 significant figures or "
    "the exact symbolic form. Example: 143.224229 not 143; 2.32625 not 2.33."
)

_FORMAT_RULES_FREEFORM = (
    "Output rules for the FINAL answer:\n"
    "- Use plain numbers (write 0.5, not '1/2 of pi'), no units, no 'x = ', "
    "no trailing punctuation.\n"
    "- " + _NUMERIC_PRECISION_RULE + "\n"
    "- " + _TOKEN_BUDGET_RULE
)

_FORMAT_RULES_MCQ = (
    "Output rules for the FINAL answer:\n"
    "- Output ONLY the letter inside \\boxed{}. Example: \\boxed{C}.\n"
    "- Do NOT write \\boxed{(C)}, \\boxed{C.}, \\boxed{C)}, or "
    "\\boxed{C: ...}. Letter only, no punctuation, no parentheses.\n"
    "- " + _TOKEN_BUDGET_RULE
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step.\n\n"
    "If the problem has K sub-answers (e.g. parts (a) (b) (c), or multiple "
    "[ANS] placeholders), use ONE of these two styles for the FINAL line:\n"
    "  (1) PREFERRED — single boxed, comma-separated:\n"
    "      \\boxed{41, 35, 16}\n"
    "  (2) Multiple boxed blocks separated ONLY by whitespace/commas:\n"
    "      \\boxed{41} \\boxed{35} \\boxed{16}\n"
    "      DO NOT put labels like '(a)', '(b)', words, or sentences "
    "BETWEEN boxed blocks — that BREAKS the parser.\n"
    "      Example of what NOT to do: '(a) \\boxed{41} (b) \\boxed{35}' — "
    "the parser only sees the last box.\n"
    "Do NOT mix the two styles in one response.\n\n"
    "If the question contains [ANS] placeholders, replace each with your "
    "boxed answer in order. The very last line of your response must be a "
    "\\boxed{...}.\n\n"
    + _FORMAT_RULES_FREEFORM
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single "
    "best answer.\n\n"
    + _FORMAT_RULES_MCQ
)


def build_prompt(question: str, options) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"\n── {label} system prompt ──")
    print(sys_p)
    print(f"\n── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "..." if len(usr_p) > 200 else "")


── MCQ system prompt ──
You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer.

Output rules for the FINAL answer:
- Output ONLY the letter inside \boxed{}. Example: \boxed{C}.
- Do NOT write \boxed{(C)}, \boxed{C.}, \boxed{C)}, or \boxed{C: ...}. Letter only, no punctuation, no parentheses.
- If you are running out of reasoning space, IMMEDIATELY output your best-guess answer inside \boxed{...} before stopping. Never end without a final boxed answer.

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form system prompt ──
You are an expert mathematician. Solve the problem step-by-step.

If the problem has K sub-answers (e.g. parts (a) (b) (c), or multiple [ANS] placeholders), use ONE of these two styles for the FINAL line:


## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.70,
    max_model_len=20480,             # 16384 generation + 4096 prompt headroom
    trust_remote_code=True,
    max_num_seqs=64,
    max_num_batched_tokens=20480,    # match max_model_len
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-03 15:35:36 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 20480, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 20480, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-03 15:35:36 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-03 15:35:36 [model.py:1680] Using max model len 20480
INFO 05-03 15:35:36 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=2990461) INFO 05-03 15:35:36 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=20480, download_dir=No

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=2990461) /home/jason/Desktop/School Works/CSE151B/151B_SP26_Competition/.venv/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=2990461)   torch._check_is_size(blocksize)


(EngineCore pid=2990461) INFO 05-03 15:35:41 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 3.007525 seconds
(EngineCore pid=2990461) INFO 05-03 15:35:43 [backends.py:1069] Using cache directory: /home/jason/.cache/vllm/torch_compile_cache/0293a04e90/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2990461) INFO 05-03 15:35:43 [backends.py:1128] Dynamo bytecode transform time: 1.37 s
(EngineCore pid=2990461) INFO 05-03 15:35:44 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 20480) from the cache, took 0.893 s
(EngineCore pid=2990461) INFO 05-03 15:35:44 [decorators.py:305] Directly load AOT compilation from path /home/jason/.cache/vllm/torch_compile_cache/torch_aot_compile/f1575bbeaf0802104a88f0d4e3efb7a369604be9e1f3e59acb3e2f7b904d777c/rank_0_0/model
(EngineCore pid=2990461) INFO 05-03 15:35:44 [monitor.py:53] torch.compile took 2.79 s in total
(EngineCore pid=2990461) INFO 05-03 15:35:44 [monitor.py:81] Initial profiling/warmup

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  89%|████████▉ | 17/19 [00:00<00:00, 19.65it/s]/home/jason/Desktop/School Works/CSE151B/151B_SP26_Competition/.venv/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=2990461)   torch._check_is_size(blocksize)
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 19/19 [00:00<00:00, 19.56it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 11/11 [00:00<00:00, 22.63it/s]


(EngineCore pid=2990461) INFO 05-03 15:35:50 [gpu_model_runner.py:6133] Graph capturing finished in 2 secs, took 0.45 GiB
(EngineCore pid=2990461) INFO 05-03 15:35:50 [gpu_worker.py:599] CUDA graph pool memory: 0.45 GiB (actual), 0.41 GiB (estimated), difference: 0.05 GiB (10.6%).
(EngineCore pid=2990461) INFO 05-03 15:35:50 [core.py:299] init engine (profile, create kv cache, warmup model) took 8.66 s (compilation: 2.79 s)
(EngineCore pid=2990461) INFO 05-03 15:35:50 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [8]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Rendering prompts:   0%|          | 0/1126 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1126 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [8]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# ── Skip scoring for private set (no `answer` field) ─────────────────────────
HAS_GOLD = data and "answer" in data[0]

results = []
if HAS_GOLD:
    sys.path.insert(0, ".")
    from judger import Judger
    judger = Judger(strict_extract=False)

    for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
        is_mcq = bool(item.get("options"))
        gold   = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "gold":     gold,
            "response": response,
            "correct":  correct,
        })
    print(f"Scoring complete. {len(results)} results.")
else:
    print("No gold answers in data — skipping scoring (private/test set).")

No gold answers in data — skipping scoring (private/test set).


## 8. Summary

Print accuracy broken down by question type.

In [9]:
if results:
    mcq_res  = [r for r in results if r["is_mcq"]]
    free_res = [r for r in results if not r["is_mcq"]]

    def acc(subset):
        return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
    print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
    print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
    print("=" * 50)
else:
    print("No scored results to summarize (private/test set).")

No scored results to summarize (private/test set).


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
import json

# ── Public-set output: full JSONL with gold + correct (re-judgeable) ───────
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

# `results` is filled by cell 22 when data has gold. If running on private set,
# fall back to writing responses only.
if results:
    rows = results
else:
    rows = [{"id": item["id"], "is_mcq": bool(item.get("options")), "response": resp}
            for item, resp in zip(data, responses)]

with open(out_path, "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

print(f"Saved {len(rows)} rows to {out_path}")

# Sanity check
boxed_marker = r"\boxed{"
total = len(rows)
empty = sum(1 for r in rows if not r.get("response"))
has_boxed = sum(1 for r in rows if boxed_marker in r.get("response", ""))
correct = sum(1 for r in rows if r.get("correct")) if "correct" in (rows[0] if rows else {}) else None
print(f"  Total rows         : {total}")
print(f"  Empty responses    : {empty}")
print(f"  Has boxed marker   : {has_boxed}")
if correct is not None:
    print(f"  Correct (judger)   : {correct}/{total}  ({correct/total*100:.2f}%)")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!